In [1]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import itertools as it

%matplotlib inline

In [3]:
class XOR(tf.Module):
    def __init__(self):
        super().__init__()
        self.built = False

    def __call__(self, x):
        if not self.built:
            input_dim = x.shape[-1]
            self.W1 = tf.Variable(tf.random.normal([input_dim, 2]), name="Weights1")
            self.b1 = tf.Variable(tf.zeros([2]), name="bais1")
            self.W2 = tf.Variable(tf.random.normal([2, 1]), name="Weights2")
            self.b2 = tf.Variable(tf.zeros([1]), name="bias2")
            self.built = True
        first = tf.sigmoid(tf.add(tf.matmul(x, self.W1), self.b1)) 
        out = tf.sigmoid(tf.add(tf.matmul(first, self.W2), self.b2))
        return out

def compute_loss(y_pred, y_true):
    return tf.reduce_mean(tf.square(y_pred - y_true))

def train_model(model, x_train, y_train, learning_rate=0.5, epochs=8000):
    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            y_pred = model(x_train) 
            loss = compute_loss(y_pred, y_train)
            
        gradients = tape.gradient(loss, model.trainable_variables)
        for var, grad in zip(model.trainable_variables, gradients):
            var.assign_sub(learning_rate * grad)
            
        if epoch % 1000 == 0:
            acc = compute_accuracy(model, x_train, y_train)
            print(f"Epoch {epoch}, Loss: {loss.numpy():.4f}, Accuracy: {acc:.4f}")


def compute_accuracy(model, x, y_true):
    y_pred = model(x)  
    y_pred_rounded = tf.round(y_pred)  
    correct = tf.equal(y_pred_rounded, y_true)
    return tf.reduce_mean(tf.cast(correct, tf.float32)).numpy()





In [5]:
xor_table = np.array([[0, 0, 0],
                      [1, 0, 1],
                      [0, 1, 1],
                      [1, 1, 0]], dtype=np.float32)

x_train = xor_table[:, :2]
y_train = xor_table[:, 2:]

model = XOR()
train_model(model, x_train, y_train)

w1 = model.W1.numpy()
w2 = model.W2.numpy()
b1 = model.b1.numpy()
b2 = model.b2.numpy()
print(f"W1:\n{model.W1.numpy()}")
print(f"b1:\n{model.b1.numpy()}")
print(f"W2:\n{model.W2.numpy()}")
print(f"b2:\n{model.b2.numpy()}")

y_pred = model(x_train).numpy().round().astype(np.uint8)
print("Predicted Truth Table:")
print(np.column_stack((xor_table[:, :2], y_pred)))



Epoch 0, Loss: 0.2637, Accuracy: 0.2500
Epoch 1000, Loss: 0.2237, Accuracy: 0.7500
Epoch 2000, Loss: 0.0128, Accuracy: 1.0000
Epoch 3000, Loss: 0.0043, Accuracy: 1.0000
Epoch 4000, Loss: 0.0025, Accuracy: 1.0000
Epoch 5000, Loss: 0.0017, Accuracy: 1.0000
Epoch 6000, Loss: 0.0013, Accuracy: 1.0000
Epoch 7000, Loss: 0.0010, Accuracy: 1.0000
W1:
[[ 5.1256447  5.919665 ]
 [-5.3331203 -5.827146 ]]
b1:
[-2.7872977  2.97465  ]
W2:
[[ 8.699059]
 [-8.196725]]
b2:
[3.8065033]
Predicted Truth Table:
[[0. 0. 0.]
 [1. 0. 1.]
 [0. 1. 1.]
 [1. 1. 0.]]
